In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import torch

RAIZ = Path("/workspace/ProtoSAM-repro")
SHA_VMAMBA = "572207a2b6553ff85d81b602eebda2041623073f01fc0c6d1280c5a2add040f6"

assert RAIZ.is_dir(), f"Repositório não encontrado: {RAIZ}"
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

arquivo_vmamba = RAIZ / "models/vmamba.py"
assert hashlib.sha256(arquivo_vmamba.read_bytes()).hexdigest() == SHA_VMAMBA

from models import vmamba

assert Path(vmamba.__file__).resolve() == arquivo_vmamba.resolve(), (
    f"Importação veio de outro local: {vmamba.__file__}"
)

CHECKPOINT_VMAMBA = RAIZ / "pretrained_model/vmamba_tiny_v2.pth"
SUPORTE_K1 = "cju3xtufwiv9c0818djsc4cqd.png"
RESOLUCAO_ENCODER = 672

assert CHECKPOINT_VMAMBA.is_file()
assert torch.cuda.is_available()

print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print("Código VMamba:", vmamba.__file__)
print("SHA256 conferido:", SHA_VMAMBA)
print("Checkpoint:", CHECKPOINT_VMAMBA.resolve())
print("Suporte:", SUPORTE_K1)
print("Resolução do encoder:", RESOLUCAO_ENCODER)

In [ ]:
CONFIG_VMAMBA = dict(
    out_indices=(3,),
    pretrained=None,  # Carregaremos os pesos explicitamente abaixo.
    depths=[2, 2, 5, 2],
    dims=96,
    drop_path_rate=0.2,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    ssm_d_state=1,
    ssm_ratio=2.0,
    ssm_dt_rank="auto",
    ssm_act_layer="silu",
    ssm_conv=3,
    ssm_conv_bias=False,
    ssm_drop_rate=0.0,
    ssm_init="v0",
    forward_type="v05_noz",
    mlp_ratio=4.0,
    mlp_act_layer="gelu",
    mlp_drop_rate=0.0,
    gmlp=False,
    patch_norm=True,
    norm_layer="ln2d",
    downsample_version="v3",
    patchembed_version="v2",
    use_checkpoint=False,
    posembed=False,
    imgsize=224,
)

with torch.device("cpu"):
    encoder = vmamba.Backbone_VSSM(**CONFIG_VMAMBA)

checkpoint = torch.load(
    CHECKPOINT_VMAMBA,
    map_location="cpu",
    weights_only=True,
)
incompatibilidades = encoder.load_state_dict(
    checkpoint["model"].copy(),
    strict=False,
)

# Aceitamos somente as diferenças já identificadas na auditoria.
ausentes_esperadas = {"outnorm3.weight", "outnorm3.bias"}
extras_esperadas = {
    "classifier.norm.weight",
    "classifier.norm.bias",
    "classifier.head.weight",
    "classifier.head.bias",
}

assert set(incompatibilidades.missing_keys) == ausentes_esperadas, (
    f"Chaves ausentes inesperadas: {incompatibilidades.missing_keys}"
)
assert set(incompatibilidades.unexpected_keys) == extras_esperadas, (
    f"Chaves extras inesperadas: {incompatibilidades.unexpected_keys}"
)

# Preserva a normalização de saída usada na integração histórica.
assert torch.equal(
    encoder.outnorm3.weight,
    torch.ones_like(encoder.outnorm3.weight),
)
assert torch.equal(
    encoder.outnorm3.bias,
    torch.zeros_like(encoder.outnorm3.bias),
)

encoder.eval()
encoder.requires_grad_(False)

assert all(not modulo.training for modulo in encoder.modules())
assert all(not parametro.requires_grad for parametro in encoder.parameters())

print("Pesos carregados com as incompatibilidades previstas.")
print("Ausentes:", incompatibilidades.missing_keys)
print("Descartadas:", incompatibilidades.unexpected_keys)
print("Normalização de saída: escala 1 e viés 0.")
print("Encoder em eval() e todos os parâmetros congelados.")
print("Parâmetros:", f"{sum(p.numel() for p in encoder.parameters()):,}")

del checkpoint

In [ ]:
import torch.nn.functional as F

# Mantém as opções numéricas registradas no baseline.
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True

encoder = encoder.cuda().eval()

gerador = torch.Generator(device="cpu").manual_seed(42)
entrada_teste = torch.randn(
    1, 3, RESOLUCAO_ENCODER, RESOLUCAO_ENCODER,
    generator=gerador,
).cuda()

with torch.inference_mode(), torch.autocast("cuda", enabled=False):
    features_nativas = encoder(entrada_teste)[-1]

    # Mesma interpolação usada na integração histórica.
    features_alp = F.interpolate(
        features_nativas,
        size=(32, 32),
        mode="bilinear",
        align_corners=False,
    )

print("Entrada:", tuple(entrada_teste.shape))
print("Características nativas:", tuple(features_nativas.shape))
print("Características para ALP:", tuple(features_alp.shape))
print("Dtype:", features_alp.dtype)
print("Valores finitos:", torch.isfinite(features_alp).all().item())

assert tuple(features_nativas.shape) == (1, 768, 21, 21)
assert tuple(features_alp.shape) == (1, 768, 32, 32)
assert features_alp.dtype == torch.float32
assert torch.isfinite(features_nativas).all().item()
assert torch.isfinite(features_alp).all().item()

del entrada_teste, features_nativas, features_alp

In [ ]:
import inspect
from pathlib import Path
from models.grid_proto_fewshot import FewShotSeg
from models.alpmodule import MultiProtoAsConv

arquivo_alp = Path(inspect.getfile(FewShotSeg)).resolve()
assert arquivo_alp == (
    RAIZ / "models/grid_proto_fewshot.py"
).resolve(), f"ALP importado de outro local: {arquivo_alp}"

print("=== Construtor do pipeline ALP ===")
print(inspect.getsource(FewShotSeg.__init__))

print("=== Construção do classificador ===")
print(inspect.getsource(FewShotSeg.get_cls))

print("=== Extração de características ===")
print(inspect.getsource(FewShotSeg.get_features))

print("=== Construtor do classificador de protótipos ===")
print(inspect.signature(MultiProtoAsConv.__init__))

In [ ]:
from copy import deepcopy
from util.consts import DEFAULT_FEATURE_SIZE

CONFIG_ALP = {
    "which_model": "vmamba_tiny",
    "cls_name": "grid_proto",
    "proto_grid_size": 8,
    "lora": 0,
    "align": False,
    "debug": False,
    "use_coco_init": False,
    "use_pos_enc": False,
    "use_slice_adapter": False,
    "adapter_layers": 3,
    "dinov2_loss": False,
}

def construir_alp_vmamba(encoder_validado, resolucao, config):
    # A classe usa o encoder recebido, sem recarregar checkpoints.
    class FewShotSegVMamba(FewShotSeg):
        def get_encoder(self):
            assert self.config["which_model"] == "vmamba_tiny"
            self.encoder = encoder_validado
            lado = max(self.image_size // 32, DEFAULT_FEATURE_SIZE)
            self.config["feature_hw"] = [lado, lado]

        def get_cls(self):
            assert self.config["cls_name"] == "grid_proto"
            grade = self.config["proto_grid_size"]
            self.cls_unit = MultiProtoAsConv(
                proto_grid=[grade, grade],
                feature_hw=self.config["feature_hw"],
                embed_dim=768,
            )

        def get_features(self, imgs_concat):
            lado_entrada = self.image_size // 32 * 32
            imgs_concat = F.interpolate(
                imgs_concat,
                size=(lado_entrada, lado_entrada),
                mode="bilinear",
                align_corners=False,
            )
            features = self.encoder(imgs_concat)[-1]

            assert features.ndim == 4 and features.shape[1] == 768
            if features.shape[-1] < DEFAULT_FEATURE_SIZE:
                features = F.interpolate(
                    features,
                    size=(DEFAULT_FEATURE_SIZE, DEFAULT_FEATURE_SIZE),
                    mode="bilinear",
                    align_corners=False,
                )
            return features

    modelo = FewShotSegVMamba(
        image_size=resolucao,
        pretrained_path=None,
        cfg=deepcopy(config),
    )
    return modelo.cuda().eval().requires_grad_(False)

alpnet = construir_alp_vmamba(
    encoder_validado=encoder,
    resolucao=RESOLUCAO_ENCODER,
    config=CONFIG_ALP,
)

assert alpnet.encoder is encoder
assert type(alpnet).forward is FewShotSeg.forward
assert all(not m.training for m in alpnet.modules())
assert all(not p.requires_grad for p in alpnet.parameters())

print("Encoder já carregado reutilizado:", alpnet.encoder is encoder)
print("Forward oficial preservado:", type(alpnet).forward is FewShotSeg.forward)
print("Grade de características:", alpnet.config["feature_hw"])
print("Kernel configurado do ALP:", alpnet.cls_unit.kernel_size)
print("Modelo em eval() e congelado.")

In [ ]:
import json

arquivo_dataset = RAIZ / "dataloaders/PolypDataset.py"
print("=== Dataset oficial ===")
print(arquivo_dataset.read_text(encoding="utf-8"))

arquivo_baseline = RAIZ / "exp/01_baseline_dinov2_k1.ipynb"
notebook_baseline = json.loads(
    arquivo_baseline.read_text(encoding="utf-8")
)

print("\n=== Uso do dataset no notebook DINOv2 ===")
for indice, celula in enumerate(notebook_baseline["cells"]):
    if celula["cell_type"] != "code":
        continue
    fonte = "".join(celula["source"])
    if "PolypDataset" in fonte:
        print(f"\n--- Célula {indice + 1} ---")
        print(fonte)

In [ ]:
import csv
import inspect
from dataloaders.PolypDataset import get_polyp_dataset
from models.segment_anything.utils.transforms import ResizeLongestSide

REFERENCIA_K1 = (
    RAIZ / "reproducao/01_avaliacao_K1_20261004T140725337013Z"
)

assert RAIZ in Path(inspect.getfile(get_polyp_dataset)).resolve().parents

with (REFERENCIA_K1 / "manifesto_teste.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    manifesto = list(csv.DictReader(arquivo))

ds_suportes, ds_teste = get_polyp_dataset(
    image_size=(1024, 1024),
    sam_trans=ResizeLongestSide(1024),
)

assert len(ds_teste) == len(manifesto) == 100

# Confere a correspondência com o manifesto, sem executar inferências.
for item in manifesto:
    indice = int(item["indice"])
    assert Path(ds_teste.images[indice]).resolve() == Path(item["imagem"]).resolve()
    assert Path(ds_teste.gts[indice]).resolve() == Path(item["mascara"]).resolve()
    assert Path(item["imagem"]).stem == Path(item["mascara"]).stem

indices_k1 = [
    i for i, caminho in enumerate(ds_suportes.images)
    if Path(caminho).name == SUPORTE_K1
]
assert len(indices_k1) == 1
indice_k1 = indices_k1[0]
assert Path(ds_suportes.gts[indice_k1]).stem == Path(SUPORTE_K1).stem

suporte = ds_suportes[indice_k1]
item_query = manifesto[0]
query = ds_teste[int(item_query["indice"])]

assert Path(item_query["imagem"]).name == "cju0u82z3cuma0835wlxrnrjv.png"
assert int((suporte["label"] > 0.5).sum()) == 51940

for nome, amostra in (("Suporte K1", suporte), ("Query", query)):
    imagem, mascara = amostra["image"], amostra["label"]
    assert tuple(imagem.shape) == (3, 1024, 1024)
    assert tuple(mascara.shape) == (1024, 1024)
    assert torch.isfinite(imagem).all().item()
    assert ((mascara == 0) | (mascara == 1)).all().item()
    print(
        f"{nome}: imagem={tuple(imagem.shape)}, "
        f"máscara={tuple(mascara.shape)}, "
        f"pixels FG={int(mascara.sum())}"
    )

print("Índice K1:", indice_k1)
print("Query:", Path(item_query["imagem"]).name)
print("Correspondência com o manifesto: conferida.")

In [ ]:
import ast
import importlib
import json

baseline = json.loads(
    (RAIZ / "exp/01_baseline_dinov2_k1.ipynb").read_text(encoding="utf-8")
)

interfaces = {}
for celula in baseline["cells"]:
    if celula["cell_type"] != "code":
        continue
    fonte = "".join(celula["source"])
    if not any(nome in fonte for nome in ("ALPNetInput", "ALPNetWrapper")):
        continue
    arvore = ast.parse(fonte)
    for no in ast.walk(arvore):
        if isinstance(no, ast.ImportFrom) and no.module and no.level == 0:
            for item in no.names:
                if item.name in ("ALPNetInput", "ALPNetWrapper"):
                    modulo = importlib.import_module(no.module)
                    interfaces[item.name] = getattr(modulo, item.name)
                    print(f"Interface: {no.module}.{item.name}")

assert set(interfaces) == {"ALPNetInput", "ALPNetWrapper"}
ALPNetInput = interfaces["ALPNetInput"]
ALPNetWrapper = interfaces["ALPNetWrapper"]

sup_img = suporte["image"].unsqueeze(0).cuda().float()
sup_mask = suporte["label"].unsqueeze(0).cuda().float()
qry = query["image"].unsqueeze(0).cuda().float()

entrada_alp = ALPNetInput(
    support_images=[sup_img],
    support_labels=[sup_mask],
    query_images=qry,
    isval=True,
    val_wsize=2,
    show_viz=False,
    supp_fts=None,
)

with torch.inference_mode(), torch.autocast("cuda", enabled=False):
    logits_alp = ALPNetWrapper(alpnet)(entrada_alp)

assert isinstance(logits_alp, torch.Tensor)
assert torch.isfinite(logits_alp).all().item()
print("Shape dos escores ALP:", tuple(logits_alp.shape))

# Aceita saída com ou sem a dimensão de batch unitário.
escores = logits_alp.detach().cpu()
if escores.ndim == 4:
    assert escores.shape[0] == 1
    escores = escores[0]
assert tuple(escores.shape) == (2, 1024, 1024)

pred_alp = escores.argmax(dim=0) == 1

# A máscara humana da query entra somente aqui, na avaliação.
gt_query = query["label"].detach().cpu() > 0.5
tp = int((pred_alp & gt_query).sum())
fp = int((pred_alp & ~gt_query).sum())
fn = int((~pred_alp & gt_query).sum())

metricas_alp = {
    "Dice": 2 * tp / (2 * tp + fp + fn),
    "IoU": tp / (tp + fp + fn),
    "precisao": tp / (tp + fp) if tp + fp else 0.0,
    "recall": tp / (tp + fn),
}

print("Pixels preditos como pólipo:", int(pred_alp.sum()))
print("TP:", tp, "| FP:", fp, "| FN:", fn)
print(json.dumps(metricas_alp, indent=2))

In [ ]:
import io
import numpy as np
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Image as DisplayImage

# Inverte a normalização apenas para visualizar o RGB.
# Não altera a entrada dos modelos.
media = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
desvio = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)

rgb = (
    query["image"].detach().cpu() * desvio + media
).clamp(0, 255).permute(1, 2, 0).numpy().astype(np.uint8)

gt = gt_query.numpy()
pred = pred_alp.numpy()

erros = np.zeros((*gt.shape, 3), dtype=np.uint8)
erros[pred & gt] = [50, 180, 80]      # TP: verde
erros[pred & ~gt] = [230, 70, 60]     # FP: vermelho
erros[~pred & gt] = [60, 120, 230]    # FN: azul

fig, eixos = plt.subplots(1, 4, figsize=(16, 4))
paineis = [
    (rgb, "Query — canvas 1024"),
    (gt, "Máscara humana"),
    (pred, f"VMamba-T + ALP\nDice = {metricas_alp['Dice']:.4f}"),
    (erros, "Acertos e erros"),
]

for eixo, (imagem, titulo) in zip(eixos, paineis):
    if imagem.ndim == 2:
        eixo.imshow(imagem, cmap="gray", vmin=0, vmax=1)
    else:
        eixo.imshow(imagem)
    eixo.set_title(titulo)
    eixo.axis("off")

fig.legend(
    handles=[
        Patch(color=np.array(cor) / 255, label=nome)
        for cor, nome in [
            ([50, 180, 80], "TP: pólipo identificado"),
            ([230, 70, 60], "FP: fundo marcado como pólipo"),
            ([60, 120, 230], "FN: pólipo não identificado"),
        ]
    ],
    loc="lower center",
    ncol=3,
)
fig.tight_layout(rect=(0, 0.12, 1, 1))

with io.BytesIO() as buffer:
    fig.savefig(buffer, format="png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))

In [ ]:
import inspect
import json
from models.ProtoSAM import ProtoSAM

print("=== Assinatura do ProtoSAM ===")
print(inspect.signature(ProtoSAM.__init__))

baseline = json.loads(
    (RAIZ / "exp/01_baseline_dinov2_k1.ipynb").read_text(encoding="utf-8")
)

print("\n=== Construção do SAM no baseline ===")
for indice, celula in enumerate(baseline["cells"]):
    if celula["cell_type"] != "code":
        continue
    fonte = "".join(celula["source"])
    if "ProtoSAM(" in fonte or "sam_model_registry[" in fonte:
        print(f"\n--- Célula {indice + 1} ---")
        print(fonte)

In [ ]:
from models.ProtoSAM import ProtoSAM, ALPNetWrapper

CHECKPOINT_SAM = RAIZ / "pretrained_model/sam_vit_h.pth"
assert CHECKPOINT_SAM.is_file()
assert "protosam" not in globals(), (
    "ProtoSAM já instanciado; não execute esta célula novamente."
)

CONFIG_SAM = {
    "image_size": [1024, 1024],
    "sam_pretrained_path": str(CHECKPOINT_SAM),
    "num_points_for_sam": 1,
    "use_points": True,
    "use_bbox": True,
    "use_mask": False,
    "debug": False,
    "use_cca": False,
    "point_mode": "both",
    "use_sam_trans": True,
    "coarse_pred_only": False,
    "alpnet_image_size": RESOLUCAO_ENCODER,
    "use_neg_points": False,
}

wrapper = ALPNetWrapper(alpnet)
protosam = ProtoSAM(
    coarse_segmentation_model=wrapper,
    **CONFIG_SAM,
).to("cuda", dtype=torch.float32)

protosam.eval().requires_grad_(False)
alpnet.eval().requires_grad_(False)

assert not any(p.requires_grad for p in alpnet.parameters())
assert not any(p.requires_grad for p in protosam.sam.parameters())
assert not alpnet.training
assert not protosam.training
assert not protosam.sam.training

print("SAM-H carregado:", CHECKPOINT_SAM.resolve())
print("Encoder: VMamba-Tiny congelado")
print("Prompts: caixa + ponto de confiança + centroide por componente")
print("CCA: False")
print("Canvas SAM: 1024 × 1024 | entrada VMamba: 672 × 672")

In [ ]:
with torch.inference_mode(), torch.autocast("cuda", enabled=False):
    predicao_sam, scores_sam = protosam(
        query_image=qry,
        coarse_model_input=entrada_alp,
        degrees_rotate=0,
    )

# Converte a saída para CPU e confere que é uma máscara binária.
if isinstance(predicao_sam, torch.Tensor):
    mascara_sam = predicao_sam.detach().cpu()
else:
    mascara_sam = torch.as_tensor(np.asarray(predicao_sam).copy())

print("Shape retornado pelo SAM:", tuple(mascara_sam.shape))
mascara_sam = mascara_sam.squeeze()

assert tuple(mascara_sam.shape) == (1024, 1024)
assert torch.isfinite(mascara_sam).all().item()
assert ((mascara_sam == 0) | (mascara_sam == 1)).all().item(), (
    f"Saída não binária: {torch.unique(mascara_sam).tolist()[:20]}"
)

pred_sam = mascara_sam.bool()
tp_sam = int((pred_sam & gt_query).sum())
fp_sam = int((pred_sam & ~gt_query).sum())
fn_sam = int((~pred_sam & gt_query).sum())

metricas_sam = {
    "Dice": 2 * tp_sam / (2 * tp_sam + fp_sam + fn_sam),
    "IoU": tp_sam / (tp_sam + fp_sam + fn_sam),
    "precisao": (
        tp_sam / (tp_sam + fp_sam) if tp_sam + fp_sam else 0.0
    ),
    "recall": tp_sam / (tp_sam + fn_sam),
}

print("Pixels preditos como pólipo:", int(pred_sam.sum()))
print("TP:", tp_sam, "| FP:", fp_sam, "| FN:", fn_sam)
print()
print(f"{'Métrica':<12} {'ALP':>10} {'ALP + SAM':>12}")
for metrica in metricas_alp:
    print(
        f"{metrica:<12} "
        f"{metricas_alp[metrica]:>10.6f} "
        f"{metricas_sam[metrica]:>12.6f}"
    )

print(
    "\nDelta Dice SAM − ALP:",
    f"{metricas_sam['Dice'] - metricas_alp['Dice']:+.6f}",
)

In [ ]:
from datetime import datetime, timezone
from PIL import Image
import hashlib
import json
import subprocess
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

execucao_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_INSPECAO = RAIZ / "reproducao" / f"vmamba_tiny_k1_inspecao_{execucao_id}"
DIR_INSPECAO.mkdir(parents=True, exist_ok=False)

def sha256_arquivo(caminho):
    resumo = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(8 * 1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()

for nome, mascara in {
    "gt": gt_query,
    "alp": pred_alp,
    "sam": pred_sam,
}.items():
    Image.fromarray(
        mascara.numpy().astype(np.uint8) * 255
    ).save(DIR_INSPECAO / f"{nome}.png")

registro = {
    "tipo": "inspecao_funcional_uma_query",
    "suporte": SUPORTE_K1,
    "query": Path(item_query["imagem"]).name,
    "config_vmamba": CONFIG_VMAMBA,
    "config_alp_efetiva": alpnet.config,
    "config_sam": CONFIG_SAM,
    "resolucao_encoder": RESOLUCAO_ENCODER,
    "normalizacao_saida": "outnorm3: escala 1, vies 0; congelada",
    "metricas": {"ALP": metricas_alp, "SAM": metricas_sam},
    "ambiente": {
        "gpu": torch.cuda.get_device_name(0),
        "python": sys.version,
        "torch": torch.__version__,
        "cuda_pytorch": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
        "tf32_cudnn": torch.backends.cudnn.allow_tf32,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    },
    "sha256": {
        "codigo_vmamba": sha256_arquivo(RAIZ / "models/vmamba.py"),
        "checkpoint_vmamba": sha256_arquivo(CHECKPOINT_VMAMBA),
        "checkpoint_sam": sha256_arquivo(CHECKPOINT_SAM),
        "imagem_suporte": sha256_arquivo(ds_suportes.images[indice_k1]),
        "mascara_suporte": sha256_arquivo(ds_suportes.gts[indice_k1]),
        "imagem_query": sha256_arquivo(item_query["imagem"]),
        "mascara_query": sha256_arquivo(item_query["mascara"]),
    },
}

registro["ambiente"]["gpu_driver"] = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip()

(DIR_INSPECAO / "registro.json").write_text(
    json.dumps(registro, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

fig, eixos = plt.subplots(1, 4, figsize=(16, 4))
paineis = [
    (rgb, "Query"),
    (gt_query.numpy(), "Máscara humana"),
    (pred_alp.numpy(), f"ALP — Dice {metricas_alp['Dice']:.4f}"),
    (pred_sam.numpy(), f"ALP + SAM — Dice {metricas_sam['Dice']:.4f}"),
]
for eixo, (imagem, titulo) in zip(eixos, paineis):
    if imagem.ndim == 2:
        eixo.imshow(imagem, cmap="gray", vmin=0, vmax=1)
    else:
        eixo.imshow(imagem)
    eixo.set_title(titulo)
    eixo.axis("off")

fig.tight_layout()
fig.savefig(DIR_INSPECAO / "comparacao.png", dpi=130, bbox_inches="tight")
plt.close(fig)

print("Salvo:", DIR_INSPECAO)
display(DisplayImage(filename=str(DIR_INSPECAO / "comparacao.png")))